# RetailMind AI — Day 14: LLM review summaries and a grounded support chatbot

Days 12 and 13 answered one question per review: positive or negative? Day 14 asks two new questions: **what do customers keep talking about**, and **how can a chatbot answer without making things up**?

This notebook reads the saved summaries from `reports/review_summaries.json`, shows the chunking and prompt steps, and runs the chatbot with a fake LLM so it works offline. The real summary was produced once with `sshleifer/distilbart-cnn-12-6`; the live LLM client is optional and only reads keys from the environment.

In [1]:
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("project folder:", os.path.basename(ROOT))

project folder: retailmind-ai


## 1. The bundled data

Thirty reviews (ten per product) and a small catalog with product details, policies and brand voice. The reviews follow the `amazon_polarity` schema: `text` plus `label` (1 positive, 0 negative), with a `product_id` added so we can group them.

In [2]:
from src.assistant.data import load_catalog, load_reviews

reviews = load_reviews()
catalog = load_catalog()
print(f"{len(reviews)} reviews, {reviews['product_id'].nunique()} products, {len(catalog['store_policies'])} store policies")
print()
for row in reviews.head(3).itertuples(index=False):
    print(f"[{row.label}] {row.text}")

30 reviews, 3 products, 4 store policies

[1] Sound is crisp and the bass is rich without muddying vocals.
[1] Battery lasts all week on a single charge, even with daily calls.
[1] The ear cushions are soft and comfortable for long work sessions.


## 2. Saved theme summaries

Running the real model downloads weights and takes time, so this cell reads the JSON that `python -m src.assistant summaries --refresh` wrote. Each product has two themes; each theme keeps the review count so the summary can be checked against the evidence.

In [3]:
import json

saved = json.load(open("reports/review_summaries.json"))
for product in saved["summaries"]:
    themes = [theme["theme"] for theme in product["themes"]]
    print(f"{product['product_id']} | {product['review_count']} reviews | themes: {themes}")
print("\nExample — RM-AUDIO-01:")
for theme in saved["summaries"][0]["themes"]:
    print(f"  {theme['theme']} ({theme['review_count']} reviews): {theme['summary']}")

RM-AUDIO-01 | 10 reviews | themes: ['What customers like', 'Common complaints']
RM-FIT-03 | 10 reviews | themes: ['What customers like', 'Common complaints']
RM-HOME-02 | 10 reviews | themes: ['What customers like', 'Common complaints']

Example — RM-AUDIO-01:
  What customers like (6 reviews): Sound is crisp and the bass is rich without muddying vocals . Battery lasts all week on a single charge, even with daily calls . Paired with laptop in seconds and switches between devices easily .
  Common complaints (4 reviews): Bluetooth connection drops intermittently when I walk away from my phone . Touch controls are too sensitive and pause the music by accident . The ear pads started peeling after a few months of daily use .


## 3. From ten reviews to one model call

A summarisation model has a word limit. `chunk_texts` joins reviews into chunks of at most 180 words, the model summarises each chunk, the summaries are tagged positive or negative, and a second pass merges each group into one theme. Ten short reviews fit into a single chunk here, so there is no information loss at this step.

In [4]:
from src.assistant.summarizer import chunk_texts

texts = reviews.loc[reviews["product_id"] == "RM-AUDIO-01", "text"].tolist()
chunks = chunk_texts(texts)
print(f"{len(texts)} reviews -> {len(chunks)} chunk(s)")
print("First 100 characters:", chunks[0][:100])

10 reviews -> 1 chunk(s)
First 100 characters: Sound is crisp and the bass is rich without muddying vocals. Battery lasts all week on a single char


## 4. The prompt is the guardrail

For the chatbot, the important idea is **grounding**: the system message contains the product and policy text, and tells the model to use only that. If the answer is not there, it must hand off to a specialist instead of guessing. A fake-news answer is worse than no answer in customer support.

In [5]:
from src.assistant.prompts import build_system_prompt

prompt = build_system_prompt("RM-AUDIO-01: 40-hour battery life. Returns: 30 days.")
print(prompt.splitlines()[0])
rules = [line for line in prompt.splitlines() if line.startswith("- ")]
print("Rules:", rules[:3])

You are the RetailMind customer support assistant.
Rules: ['- Answer only from the approved product and policy information below.', '- Never invent prices, features, policies or delivery times.', '- If the answer is not in the information, say exactly: "I don\'t have that information in our product details. Let me connect you with a support specialist."']


## 5. Retrieval plus history makes a conversation

`retrieve_context` picks only the catalog lines that mention the product or the words in the question, so the prompt stays small. `SupportChatbot` keeps the last ten turns so you can say "and the return policy?" without repeating the product. Here a `FakeLLM` stands in for the API, so this cell needs no key or network.

In [6]:
from src.assistant.chatbot import SupportChatbot

class FakeLLM:
    def __init__(self):
        self.calls = []

    def chat(self, messages, **kwargs):
        self.calls.append(messages)
        return "The headphones last about 40 hours on one charge."

llm = FakeLLM()
bot = SupportChatbot(llm=llm)
print("Bot:", bot.ask("How long does the battery last?", product_id="RM-AUDIO-01"))
print(f"Turns after one question: {len(bot.conversation_history)} | ground truth in prompt: {'40-hour battery' in llm.calls[0][0]['content']}")
bot.reset()
print("Turns after reset:", len(bot.conversation_history))

Bot: The headphones last about 40 hours on one charge.
Turns after one question: 2 | ground truth in prompt: True
Turns after reset: 0


## 6. The live LLM client is optional and key-safe

`LLMClient` reads `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` from the environment and never prints or stores the value in logs. Without a key it raises a clear setup error. Passing a `transport` keeps the client testable without spending money, which is what the 19 offline tests do.

In [7]:
from src.assistant.client import LLMClient, LLMConfigurationError

try:
    LLMClient(environ={})
except LLMConfigurationError as exc:
    print("No key ->", str(exc)[:60])

client = LLMClient(environ={"OPENAI_API_KEY": "example-key"}, transport=lambda **kw: "offline reply")
reply = client.chat([{"role": "user", "content": "Hi"}])
print(f"provider: {client.provider} | model: {client.model} | reply: {reply} | key hidden: {'example-key' not in repr(client)}")

No key -> No LLM provider configured. Set OPENAI_API_KEY or ANTHROPIC_
provider: openai | model: gpt-4o-mini | reply: offline reply | key hidden: True


## Try it yourself

From the project root:

```bash
python -m src.assistant summaries
python -m src.assistant summaries --refresh
python -m src.assistant ask "What do owners complain about?" --product RM-AUDIO-01
python -m src.assistant chat --product RM-AUDIO-01
python -m unittest discover -s tests
```

For a real answer, set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` in a `.env` file (copy `.env.example`) and optionally `RETAILMIND_LLM_PROVIDER` / `RETAILMIND_LLM_MODEL`. Without a key, the CLI reports the setup error instead of failing silently.

**Be honest about the limits:** the summaries come from 30 short reviews, the themes are assigned by keyword rules, the chatbot only knows the tiny bundled catalog, and no factual claim is checked against live systems. This is a working pattern, not a production support agent.